# Code to AI — Week 6, Day 2: Duplicates, Types & Text Cleanup
**Pandas II — Data Cleaning**

---

Welcome back! Yesterday we fixed missing values — the gaps in our data. Today we fix the values that *are* there but are wrong: rows recorded twice, numbers stored as text, dates that pandas does not recognise as dates, and the same word spelled five different ways.

These problems are quieter than missing values. Nothing looks empty, so nothing looks broken — until your counts come out wrong.

**By the end of this notebook you will be able to:**
- Find and remove duplicate rows
- Check and convert column data types
- Turn text into real dates and pull out the year and month
- Clean up messy text with the `.str` tools
- Merge inconsistent category spellings into one
- Rename columns into a clean, consistent style

## 0. A deliberately messy dataset

Real messy data is hard to show in a classroom because the mess is scattered across
thousands of rows. So let's build a tiny table with every problem packed into 5 rows,
where you can see all of it at once.

Run the cell and study the table carefully before reading on.

In [60]:
import pandas as pd
import seaborn as sns

messy = pd.DataFrame({
    'Customer Name': ['  Ali Hasnain', 'sara khan', 'AHMED RAZA', 'sara khan', 'Zainab  Ali'],
    'Country':       ['pakistan', 'Pakistan ', 'PAKISTAN', 'Pakistan ', ' india'],
    'Order Date':    ['2026-01-15', '2026-02-03', '2026-02-20', '2026-02-03', '2026-03-11'],
    'Amount':        ['1500', '2300', 'not available', '2300', '900'],
    'Quantity':      [2, 3, 1, 3, 5]
})

messy

,Customer Name,Country,Order Date,Amount,Quantity
0,Ali Hasnain,pakistan,2026-01-15,1500,2
1,sara khan,Pakistan,2026-02-03,2300,3
2,AHMED RAZA,PAKISTAN,2026-02-20,not available,1
3,sara khan,Pakistan,2026-02-03,2300,3
4,Zainab Ali,india,2026-03-11,900,5


Take a moment. How many problems can you spot?

Here is the full list:

1. `'  Ali Hasnain'` and `'Zainab  Ali'` have **extra spaces**
2. Names use inconsistent capitalisation: `'sara khan'` vs `'AHMED RAZA'`
3. `Country` has `'pakistan'`, `'Pakistan '`, `'PAKISTAN'` — the same country, three ways
4. Row 1 and row 3 are an **exact duplicate** (sara khan, same date, same amount)
5. `Order Date` is **text**, not a date
6. `Amount` is **text** because one value says `'not available'`
7. Column names have **spaces and capitals**, which are awkward to type

Let's fix them one at a time.

---
## 1. Duplicates

### 1.1 Finding them

`.duplicated()` marks each row `True` if it is an exact copy of a row that appeared earlier.
The first occurrence is `False` — only the repeats are flagged.

In [61]:
print(messy.duplicated())

0    False
1    False
2    False
3     True
4    False
dtype: bool


In [62]:
print("Number of duplicate rows:", messy.duplicated().sum())

Number of duplicate rows: 1


To actually *see* which rows are duplicated, use the result as a filter — exactly the
boolean filtering you learned in Week 5.

In [63]:
messy[messy.duplicated()]

,Customer Name,Country,Order Date,Amount,Quantity
3,sara khan,Pakistan,2026-02-03,2300,3


### 1.2 Removing them

`.drop_duplicates()` keeps the first occurrence and removes the later copies.

In [64]:
no_dupes = messy.drop_duplicates()

print("Before:", len(messy), "rows")
print("After: ", len(no_dupes), "rows")
no_dupes

Before: 5 rows
After:  4 rows


,Customer Name,Country,Order Date,Amount,Quantity
0,Ali Hasnain,pakistan,2026-01-15,1500,2
1,sara khan,Pakistan,2026-02-03,2300,3
2,AHMED RAZA,PAKISTAN,2026-02-20,not available,1
4,Zainab Ali,india,2026-03-11,900,5


In [65]:
messy

,Customer Name,Country,Order Date,Amount,Quantity
0,Ali Hasnain,pakistan,2026-01-15,1500,2
1,sara khan,Pakistan,2026-02-03,2300,3
2,AHMED RAZA,PAKISTAN,2026-02-20,not available,1
3,sara khan,Pakistan,2026-02-03,2300,3
4,Zainab Ali,india,2026-03-11,900,5


### 1.3 `subset=` — duplicates on specific columns

Sometimes a row is not an exact copy, but it is still a duplicate *for your purposes*.

If a customer should only appear once, check for duplicates on the name column alone:

In [66]:
print("Duplicate names:", messy.duplicated(subset=['Customer Name']).sum())

Duplicate names: 1


### 1.4 `keep=` — which copy to keep

- `keep='first'` (default) — keep the first, drop the rest
- `keep='last'` — keep the last, drop the earlier ones
- `keep=False` — drop **every** copy including the first

`keep='last'` is useful when later rows are more recent updates.
`keep=False` is useful when you want to *inspect* all duplicates rather than clean them.

In [67]:
messy

,Customer Name,Country,Order Date,Amount,Quantity
0,Ali Hasnain,pakistan,2026-01-15,1500,2
1,sara khan,Pakistan,2026-02-03,2300,3
2,AHMED RAZA,PAKISTAN,2026-02-20,not available,1
3,sara khan,Pakistan,2026-02-03,2300,3
4,Zainab Ali,india,2026-03-11,900,5


In [68]:
print("keep='first' leaves:", len(messy.drop_duplicates(keep='first')), "rows")
print("keep='last' leaves: ", len(messy.drop_duplicates(keep='last')), "rows")
print("keep=False leaves:  ", len(messy.drop_duplicates(keep=False)), "rows")

keep='first' leaves: 4 rows
keep='last' leaves:  4 rows
keep=False leaves:   3 rows


In [69]:
messy[messy.duplicated(keep='first')]

,Customer Name,Country,Order Date,Amount,Quantity
3,sara khan,Pakistan,2026-02-03,2300,3


In [70]:
messy[messy.duplicated(keep='last')]

,Customer Name,Country,Order Date,Amount,Quantity
1,sara khan,Pakistan,2026-02-03,2300,3


In [ ]:
messy[messy.duplicated(keep=False)] # use for inspection

,Customer Name,Country,Order Date,Amount,Quantity
1,sara khan,Pakistan,2026-02-03,2300,3
3,sara khan,Pakistan,2026-02-03,2300,3


Notice `keep=False` leaves 3 rows, not 4 — because it removed **both** copies of the
duplicated row, not just one. Use it carefully.

Let's carry on with the properly de-duplicated version.

In [72]:
clean = messy.drop_duplicates().copy()
print("Working with", len(clean), "rows")

Working with 4 rows


In [73]:
clean

,Customer Name,Country,Order Date,Amount,Quantity
0,Ali Hasnain,pakistan,2026-01-15,1500,2
1,sara khan,Pakistan,2026-02-03,2300,3
2,AHMED RAZA,PAKISTAN,2026-02-20,not available,1
4,Zainab Ali,india,2026-03-11,900,5


---
## 2. Data types

### 2.1 Checking types with `.dtypes`

Every column has a type. `object` (or `str`) means text. `int64` and `float64` are numbers.

In [74]:
clean.info()

<class 'pandas.DataFrame'>
Index: 4 entries, 0 to 4
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Customer Name  4 non-null      str  
 1   Country        4 non-null      str  
 2   Order Date     4 non-null      str  
 3   Amount         4 non-null      str  
 4   Quantity       4 non-null      int64
dtypes: int64(1), str(4)
memory usage: 192.0 bytes


In [75]:
print(clean.dtypes)

Customer Name      str
Country            str
Order Date         str
Amount             str
Quantity         int64
dtype: object


`Amount` is text. That is a real problem — you cannot do maths on text.

Watch what happens if we try to add it up:

In [76]:
# This "works" but gives nonsense - it glues the text together instead of adding
print(clean['Amount'].sum())

15002300not available900


That is string concatenation, not addition. `'1500' + '2300'` becomes `'15002300'`.

This is exactly the kind of silent error that ruins an analysis: no crash, no warning,
just a wrong answer.

### 2.2 `.astype()` — the direct conversion

`.astype()` converts a column to a different type. It works well when the data is clean.

In [77]:
# Quantity is already a number, but let's convert it to float to see .astype() work
print("Before:", clean['Quantity'].dtype)

example = clean['Quantity'].astype(float)
print("After: ", example.dtype)
print(example.tolist())

Before: int64
After:  float64
[2.0, 3.0, 1.0, 5.0]


In [78]:
clean

,Customer Name,Country,Order Date,Amount,Quantity
0,Ali Hasnain,pakistan,2026-01-15,1500,2
1,sara khan,Pakistan,2026-02-03,2300,3
2,AHMED RAZA,PAKISTAN,2026-02-20,not available,1
4,Zainab Ali,india,2026-03-11,900,5


But `.astype()` fails the moment it meets something it cannot convert.
Our `Amount` column contains the text `'not available'`:

In [79]:
check = clean['Amount'].astype(int)

ValueError: invalid literal for int() with base 10: 'not available'

In [80]:
# This will raise an error - that is the point of this cell
try:
    clean['Amount'].astype(int)
except ValueError as e:
    print("ValueError:", e)

ValueError: invalid literal for int() with base 10: 'not available'


In [81]:
clean

,Customer Name,Country,Order Date,Amount,Quantity
0,Ali Hasnain,pakistan,2026-01-15,1500,2
1,sara khan,Pakistan,2026-02-03,2300,3
2,AHMED RAZA,PAKISTAN,2026-02-20,not available,1
4,Zainab Ali,india,2026-03-11,900,5


### 2.3 `pd.to_numeric(errors='coerce')` — the forgiving conversion

`pd.to_numeric()` with `errors='coerce'` converts what it can, and turns anything
it cannot convert into `NaN` (missing).

This is a lovely trick: it takes a **type problem** and turns it into a **missing-data
problem** — and you already know how to solve those from yesterday.

In [82]:
clean['Amount'] = pd.to_numeric(clean['Amount'], errors='coerce')

print(clean['Amount'])
print()
print("Type now:", clean['Amount'].dtype)
print("Missing values created:", clean['Amount'].isnull().sum())

0    1500.0
1    2300.0
2       NaN
4     900.0
Name: Amount, dtype: float64

Type now: float64
Missing values created: 1


`'not available'` became `NaN`. Now we handle it with yesterday's tools:

In [84]:
clean['Amount'].median()

np.float64(1500.0)

In [83]:
clean['Amount'] = clean['Amount'].fillna(clean['Amount'].median())

print(clean['Amount'])
print()
print("Total sales:", clean['Amount'].sum())

0    1500.0
1    2300.0
2    1500.0
4     900.0
Name: Amount, dtype: float64

Total sales: 6200.0


Now `.sum()` gives a real number instead of glued-together text.

---
## 3. Dates

### 3.1 `pd.to_datetime()`

Dates stored as text are just strings — pandas has no idea they mean anything.
`pd.to_datetime()` converts them into real dates.

In [85]:
clean

,Customer Name,Country,Order Date,Amount,Quantity
0,Ali Hasnain,pakistan,2026-01-15,1500.0,2
1,sara khan,Pakistan,2026-02-03,2300.0,3
2,AHMED RAZA,PAKISTAN,2026-02-20,1500.0,1
4,Zainab Ali,india,2026-03-11,900.0,5


In [86]:
clean.info()

<class 'pandas.DataFrame'>
Index: 4 entries, 0 to 4
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Customer Name  4 non-null      str    
 1   Country        4 non-null      str    
 2   Order Date     4 non-null      str    
 3   Amount         4 non-null      float64
 4   Quantity       4 non-null      int64  
dtypes: float64(1), int64(1), str(3)
memory usage: 192.0 bytes


In [87]:
print("Before:", clean['Order Date'].dtype)

clean['Order Date'] = pd.to_datetime(clean['Order Date'])

print("After: ", clean['Order Date'].dtype)
print()
print(clean['Order Date'])

Before: str
After:  datetime64[us]

0   2026-01-15
1   2026-02-03
2   2026-02-20
4   2026-03-11
Name: Order Date, dtype: datetime64[us]


In [88]:
clean

,Customer Name,Country,Order Date,Amount,Quantity
0,Ali Hasnain,pakistan,2026-01-15,1500.0,2
1,sara khan,Pakistan,2026-02-03,2300.0,3
2,AHMED RAZA,PAKISTAN,2026-02-20,1500.0,1
4,Zainab Ali,india,2026-03-11,900.0,5


In [89]:
clean.info()

<class 'pandas.DataFrame'>
Index: 4 entries, 0 to 4
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Customer Name  4 non-null      str           
 1   Country        4 non-null      str           
 2   Order Date     4 non-null      datetime64[us]
 3   Amount         4 non-null      float64       
 4   Quantity       4 non-null      int64         
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 192.0 bytes


### 3.2 `.dt` — pulling pieces out of a date

Once a column is a real date, the `.dt` accessor unlocks its parts.
This is where the conversion pays off.

In [90]:
clean['year'] = clean['Order Date'].dt.year
clean['month'] = clean['Order Date'].dt.month
clean['month_name'] = clean['Order Date'].dt.month_name()
clean['day_name'] = clean['Order Date'].dt.day_name()

clean[['Order Date', 'year', 'month', 'month_name', 'day_name']]

,Order Date,year,month,month_name,day_name
0,2026-01-15,2026,1,January,Thursday
1,2026-02-03,2026,2,February,Tuesday
2,2026-02-20,2026,2,February,Friday
4,2026-03-11,2026,3,March,Wednesday


Extracting a `year` column like this is one of the most useful moves in data analysis —
it lets you count and compare things per year, which we will do in the project on Day 3.

---
## 4. Cleaning text with `.str`

Text columns get a special toolkit through `.str`. Every normal Python string method
you know has a `.str` version that applies to the whole column at once.

### 4.1 `.str.strip()` — remove surrounding spaces

Leading and trailing spaces are invisible on screen but break everything.
`'Pakistan '` and `'Pakistan'` are two different values to a computer.

In [91]:
clean

,Customer Name,Country,Order Date,Amount,Quantity,year,month,month_name,day_name
0,Ali Hasnain,pakistan,2026-01-15,1500.0,2,2026,1,January,Thursday
1,sara khan,Pakistan,2026-02-03,2300.0,3,2026,2,February,Tuesday
2,AHMED RAZA,PAKISTAN,2026-02-20,1500.0,1,2026,2,February,Friday
4,Zainab Ali,india,2026-03-11,900.0,5,2026,3,March,Wednesday


In [92]:
# Proof that the spaces matter
print("Unique countries BEFORE cleaning:")
print(clean['Country'].unique())
print()
print("Count of unique values:", clean['Country'].nunique())

Unique countries BEFORE cleaning:
<StringArray>
['pakistan', 'Pakistan ', 'PAKISTAN', ' india']
Length: 4, dtype: str

Count of unique values: 4


In [93]:
clean['Country'] = clean['Country'].str.strip()

print("After .str.strip():")
print(clean['Country'].unique())

After .str.strip():
<StringArray>
['pakistan', 'Pakistan', 'PAKISTAN', 'india']
Length: 4, dtype: str


In [94]:
print(clean['Country'].unique())

<StringArray>
['pakistan', 'Pakistan', 'PAKISTAN', 'india']
Length: 4, dtype: str


Better — the spaces are gone. But `'pakistan'`, `'Pakistan'` and `'PAKISTAN'` are still
three separate values because of capitalisation.

### 4.2 `.str.lower()`, `.str.upper()`, `.str.title()`

Making everything the same case is how you merge them.

- `.str.lower()` → `pakistan`
- `.str.upper()` → `PAKISTAN`
- `.str.title()` → `Pakistan` (first letter of each word capitalised)

`.str.title()` is usually the nicest for names and places.

In [95]:
clean['Country'] = clean['Country'].str.title()

print(clean['Country'].unique())
print("Count of unique values:", clean['Country'].nunique())

<StringArray>
['Pakistan', 'India']
Length: 2, dtype: str
Count of unique values: 2


In [96]:
clean

,Customer Name,Country,Order Date,Amount,Quantity,year,month,month_name,day_name
0,Ali Hasnain,Pakistan,2026-01-15,1500.0,2,2026,1,January,Thursday
1,sara khan,Pakistan,2026-02-03,2300.0,3,2026,2,February,Tuesday
2,AHMED RAZA,Pakistan,2026-02-20,1500.0,1,2026,2,February,Friday
4,Zainab Ali,India,2026-03-11,900.0,5,2026,3,March,Wednesday


From 4 apparent countries down to the 2 that actually exist in this data.

**This is the single most important idea in today's class.** Before cleaning, a count of
countries would have been badly wrong. Nothing was missing, nothing crashed — the answer
was just quietly incorrect.

### 4.3 Chaining it together

`.str` methods can be chained, which is how you will usually write this in practice.

In [97]:
clean['Customer Name'] = clean['Customer Name'].str.strip().str.title()

print(clean['Customer Name'].tolist())

['Ali Hasnain', 'Sara Khan', 'Ahmed Raza', 'Zainab  Ali']


### 4.4 `.str.replace()` — fixing what is left

`'Zainab  Ali'` still has a double space in the middle, which `.strip()` does not touch
because it only handles the *ends* of a string.

In [98]:
clean['Customer Name'].unique()

<StringArray>
['Ali Hasnain', 'Sara Khan', 'Ahmed Raza', 'Zainab  Ali']
Length: 4, dtype: str

In [99]:
clean['Customer Name'] = clean['Customer Name'].str.replace('  ', ' ')

print(clean['Customer Name'].tolist())

['Ali Hasnain', 'Sara Khan', 'Ahmed Raza', 'Zainab Ali']


### 4.5 The payoff: `.value_counts()` on clean data

Here is the whole lesson in one comparison — the same question asked of the dirty column
and the clean column.

In [100]:
print("DIRTY data - country counts:")
print(messy['Country'].value_counts())
print()
print("CLEAN data - country counts:")
print(clean['Country'].value_counts())

DIRTY data - country counts:
Country
Pakistan     2
pakistan     1
PAKISTAN     1
 india       1
Name: count, dtype: int64

CLEAN data - country counts:
Country
Pakistan    3
India       1
Name: count, dtype: int64


The dirty version reports Pakistan split across three different spellings. The clean
version tells you the truth.

---
## 5. Renaming columns

### 5.1 `.rename()` — specific columns

`Customer Name` with a space and capitals is awkward to type. Let's fix the names.

In [101]:
clean

,Customer Name,Country,Order Date,Amount,Quantity,year,month,month_name,day_name
0,Ali Hasnain,Pakistan,2026-01-15,1500.0,2,2026,1,January,Thursday
1,Sara Khan,Pakistan,2026-02-03,2300.0,3,2026,2,February,Tuesday
2,Ahmed Raza,Pakistan,2026-02-20,1500.0,1,2026,2,February,Friday
4,Zainab Ali,India,2026-03-11,900.0,5,2026,3,March,Wednesday


In [102]:
clean = clean.rename(columns={
    'Customer Name': 'customer_name',
    'Order Date': 'order_date'
})

print(list(clean.columns))

['customer_name', 'Country', 'order_date', 'Amount', 'Quantity', 'year', 'month', 'month_name', 'day_name']


In [103]:
clean.columns

Index(['customer_name', 'Country', 'order_date', 'Amount', 'Quantity', 'year',
       'month', 'month_name', 'day_name'],
      dtype='str')

### 5.2 Cleaning all column names at once

For a wide dataset, renaming one by one is tedious. `.columns` is itself a text collection,
so the same `.str` tools work on it.

The standard style is lowercase with underscores instead of spaces — often called
`snake_case`.

In [104]:
clean.columns = clean.columns.str.lower().str.replace(' ', '_')

print(list(clean.columns))

['customer_name', 'country', 'order_date', 'amount', 'quantity', 'year', 'month', 'month_name', 'day_name']


---
## 6. Splitting a column

`.str.split()` breaks one column into pieces. With `expand=True` the pieces become
separate columns.

We will need this on Day 3 for the Netflix `duration` column, which stores values like
`"90 min"` — a number and a unit stuck together in one text value.

Let's practise on a small example.

In [105]:
durations = pd.DataFrame({
    'title': ['Movie A', 'Movie B', 'Show C', 'Show D'],
    'duration': ['90 min', '142 min', '2 Seasons', '1 Season']
})

durations

,title,duration
0,Movie A,90 min
1,Movie B,142 min
2,Show C,2 Seasons
3,Show D,1 Season


In [106]:
durations.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   title     4 non-null      str  
 1   duration  4 non-null      str  
dtypes: str(2)
memory usage: 196.0 bytes


In [107]:
# Split on the space, into two new columns
parts = durations['duration'].str.split(' ', expand=True)
parts

,0,1
0,90,min
1,142,min
2,2,Seasons
3,1,Season


In [108]:
durations['duration_number'] = parts[0]
durations['duration_unit'] = parts[1]

durations

,title,duration,duration_number,duration_unit
0,Movie A,90 min,90,min
1,Movie B,142 min,142,min
2,Show C,2 Seasons,2,Seasons
3,Show D,1 Season,1,Season


In [109]:
durations.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   title            4 non-null      str  
 1   duration         4 non-null      str  
 2   duration_number  4 non-null      str  
 3   duration_unit    4 non-null      str  
dtypes: str(4)
memory usage: 260.0 bytes


`duration_number` is still text. One more step makes it usable:

In [110]:
durations['duration_number'] = pd.to_numeric(durations['duration_number'])

print(durations.dtypes)
print()
durations

title                str
duration             str
duration_number    int64
duration_unit        str
dtype: object



,title,duration,duration_number,duration_unit
0,Movie A,90 min,90,min
1,Movie B,142 min,142,min
2,Show C,2 Seasons,2,Seasons
3,Show D,1 Season,1,Season


Now you could filter for movies longer than 100 minutes — impossible before the split.

---
## 7. The finished table

Let's look at what we started with and what we ended with.

In [111]:
print("=== BEFORE ===")
print(messy)
print()
print("Rows:", len(messy), "| Unique countries:", messy['Country'].nunique())

=== BEFORE ===
   Customer Name    Country  Order Date         Amount  Quantity
0    Ali Hasnain   pakistan  2026-01-15           1500         2
1      sara khan  Pakistan   2026-02-03           2300         3
2     AHMED RAZA   PAKISTAN  2026-02-20  not available         1
3      sara khan  Pakistan   2026-02-03           2300         3
4    Zainab  Ali      india  2026-03-11            900         5

Rows: 5 | Unique countries: 4


In [112]:
print("=== AFTER ===")
print(clean[['customer_name', 'country', 'order_date', 'amount', 'quantity']])
print()
print("Rows:", len(clean), "| Unique countries:", clean['country'].nunique())
print()
print("Data types:")
print(clean.dtypes)

=== AFTER ===
  customer_name   country order_date  amount  quantity
0   Ali Hasnain  Pakistan 2026-01-15  1500.0         2
1     Sara Khan  Pakistan 2026-02-03  2300.0         3
2    Ahmed Raza  Pakistan 2026-02-20  1500.0         1
4    Zainab Ali     India 2026-03-11   900.0         5

Rows: 4 | Unique countries: 2

Data types:
customer_name               str
country                     str
order_date       datetime64[us]
amount                  float64
quantity                  int64
year                      int32
month                     int32
month_name                  str
day_name                    str
dtype: object


---
## 8. Trying it on real data

Our toy table made the problems visible. Now let's confirm the same tools work on a
real dataset — `taxis`, which has 6,433 New York taxi trips.

In [113]:
taxis = sns.load_dataset('taxis')

print("Shape:", taxis.shape)
print()
print("Duplicate rows:", taxis.duplicated().sum())
print()
print(taxis.dtypes)

Shape: (6433, 14)

Duplicate rows: 0

pickup             datetime64[us]
dropoff            datetime64[us]
passengers                  int64
distance                  float64
fare                      float64
tip                       float64
tolls                     float64
total                     float64
color                         str
payment                       str
pickup_zone                   str
dropoff_zone                  str
pickup_borough                str
dropoff_borough               str
dtype: object


In [114]:
taxis.head()

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


In [115]:
# The pickup column is already a proper date, so .dt works straight away
taxis_clean = taxis.copy()
taxis_clean['pickup_month'] = taxis_clean['pickup'].dt.month_name()
taxis_clean['pickup_day'] = taxis_clean['pickup'].dt.day_name()

print(taxis_clean['pickup_day'].value_counts())

pickup_day
Friday       1115
Saturday     1046
Wednesday     966
Thursday      905
Sunday        868
Tuesday       825
Monday        708
Name: count, dtype: int64


In [116]:
taxis_clean.head()

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough,pickup_month,pickup_day
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan,March,Saturday
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan,March,Monday
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan,March,Wednesday
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan,March,Sunday
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan,March,Saturday


In [117]:
# Standardise the text columns, exactly as we did on the toy data
print("Payment types before:", taxis_clean['payment'].unique())

taxis_clean['payment'] = taxis_clean['payment'].str.strip().str.title()

print("Payment types after: ", taxis_clean['payment'].unique())

Payment types before: <StringArray>
['credit card', 'cash', nan]
Length: 3, dtype: str
Payment types after:  <StringArray>
['Credit Card', 'Cash', nan]
Length: 3, dtype: str


In [118]:
# And clean up the column names in one line
taxis_clean.columns = taxis_clean.columns.str.lower().str.replace(' ', '_')
print(list(taxis_clean.columns))

['pickup', 'dropoff', 'passengers', 'distance', 'fare', 'tip', 'tolls', 'total', 'color', 'payment', 'pickup_zone', 'dropoff_zone', 'pickup_borough', 'dropoff_borough', 'pickup_month', 'pickup_day']


In [ ]:
for i in clean.columns:
    print(i)
    

customer_name
country
order_date
amount
quantity
year
month
month_name
day_name


---
## 9. Your cleaning checklist

From now on, run through this list every time you open a new dataset:

| Step | Code |
|------|------|
| 1. Check for duplicates | `df.duplicated().sum()` |
| 2. Remove them | `df.drop_duplicates()` |
| 3. Check types | `df.dtypes` |
| 4. Fix numbers stored as text | `pd.to_numeric(df['col'], errors='coerce')` |
| 5. Fix dates stored as text | `pd.to_datetime(df['col'])` |
| 6. Extract date parts | `df['col'].dt.year` |
| 7. Strip spaces from text | `df['col'].str.strip()` |
| 8. Standardise case | `df['col'].str.title()` |
| 9. Tidy column names | `df.columns.str.lower().str.replace(' ', '_')` |
| 10. Check the result | `df['col'].value_counts()` |

Combine this with yesterday's missing-value work and you have a complete cleaning workflow.

**Tomorrow:** we use all of it on a real, genuinely messy Netflix dataset — and ship the
project to GitHub. 🚀